# 01 - Extract and Profile Source Data

This notebook reads the existing operational CSV files from `data/raw` and profiles their structure before transformation. Raw data is treated as fixed source input, so this notebook does not regenerate or overwrite source CSV files.


## Setup Project Paths

Prepare reusable project paths for the notebook workflow. The path logic works whether the notebook is opened from the project root or from inside the `notebooks` folder.

**Input:** current working directory.  
**Output:** paths for `data/raw`, `data/processed`, `data/validation`, and `output`.


In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
VALIDATION_DIR = PROJECT_ROOT / 'data' / 'validation'
OUTPUT_DIR = PROJECT_ROOT / 'output'

for directory in [RAW_DIR, PROCESSED_DIR, VALIDATION_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Raw data directory:', RAW_DIR)


Project root: D:\Semester 8\Data Warehouse\erajaya-data-warehouse
Raw data directory: D:\Semester 8\Data Warehouse\erajaya-data-warehouse\data\raw


## Register Required Raw Files

Define the required operational source files before the pipeline continues. This check makes missing source files easy to identify before the transformation notebook is executed.

**Input:** CSV files in `data/raw`.  
**Output:** validated list of raw source files.


In [2]:
required_raw_files = [
    'customers.csv',
    'products.csv',
    'stores.csv',
    'promotions.csv',
    'sales_transactions.csv',
    'sales_details.csv',
    'payments.csv',
    'inventory.csv',
]

missing_files = [filename for filename in required_raw_files if not (RAW_DIR / filename).exists()]
if missing_files:
    raise FileNotFoundError(
        'Missing raw source files in data/raw: ' + ', '.join(missing_files) +
        '. Restore these files before running the extract/profile notebook.'
    )

pd.DataFrame({'raw_file': required_raw_files})


,raw_file
0,customers.csv
1,products.csv
2,stores.csv
3,promotions.csv
4,sales_transactions.csv
5,sales_details.csv
6,payments.csv
7,inventory.csv


## Load Raw Source Data

Read each raw CSV file into the `raw_frames` dictionary. The data is loaded only for inspection and profiling, so the original files in `data/raw` remain unchanged.

**Input:** eight raw CSV files.  
**Output:** `raw_frames`, a dictionary of source DataFrames.


In [3]:
raw_frames = {filename: pd.read_csv(RAW_DIR / filename) for filename in required_raw_files}

for filename, df in raw_frames.items():
    print(f'{filename}: {len(df):,} rows, {len(df.columns):,} columns')


customers.csv: 1,000 rows, 11 columns
products.csv: 100 rows, 9 columns
stores.csv: 25 rows, 7 columns
promotions.csv: 21 rows, 6 columns
sales_transactions.csv: 2,000 rows, 9 columns
sales_details.csv: 4,688 rows, 7 columns
payments.csv: 2,000 rows, 5 columns
inventory.csv: 2,500 rows, 7 columns


## Profile Raw Data

Create an initial source profile for each raw file, including row count, column count, full-row duplicates, missing values, and available columns. The result helps confirm that the source data is ready for transformation.

**Input:** `raw_frames`.  
**Output:** `data/validation/extract_profile_summary.csv`.


In [4]:
profile_rows = []
for filename, df in raw_frames.items():
    profile_rows.append({
        'file_name': filename,
        'row_count': len(df),
        'column_count': len(df.columns),
        'duplicate_row_count': int(df.duplicated().sum()),
        'missing_value_count': int(df.isna().sum().sum()),
        'columns': ', '.join(df.columns),
        'is_extract_valid': len(df) > 0 and len(df.columns) > 0,
    })

extract_profile_summary = pd.DataFrame(profile_rows)
extract_profile_summary.to_csv(VALIDATION_DIR / 'extract_profile_summary.csv', index=False)
extract_profile_summary


,file_name,row_count,column_count,duplicate_row_count,missing_value_count,columns,is_extract_valid
0,customers.csv,1000,11,0,5,"customer_id, customer_name, gender, birth_date...",True
1,products.csv,100,9,0,0,"product_id, product_name, brand, category, sub...",True
2,stores.csv,25,7,0,0,"store_id, store_name, store_type, province, ci...",True
3,promotions.csv,21,6,0,0,"promotion_id, promotion_name, promotion_type, ...",True
4,sales_transactions.csv,2000,9,0,943,"transaction_id, transaction_date, customer_id,...",True
5,sales_details.csv,4688,7,0,0,"detail_id, transaction_id, product_id, promoti...",True
6,payments.csv,2000,5,0,0,"payment_id, payment_method, payment_status, pa...",True
7,inventory.csv,2500,7,0,0,"inventory_id, snapshot_date, store_id, product...",True


## Inspect Sample Rows

Preview the first rows of each source file so the team can quickly inspect the operational data structure before moving to the transformation notebook.

**Input:** `raw_frames`.  
**Output:** source data previews inside the notebook.


In [5]:
for filename, df in raw_frames.items():
    print(f'\n{filename}')
    display(df.head(3))



customers.csv


,customer_id,customer_name,gender,birth_date,email,phone,loyalty_tier,customer_segment,province,city,registered_date
0,CUST-GUEST,Guest Customer,Unknown,NaN,NaN,NaN,Regular,Walk-in,NaN,NaN,2023-01-01
1,CUST-0001,Dewi Pratama,Male,1982-03-24,dewi.pratama1@example.com,8.334233e+10,Silver,Student,Sumatera Utara,Deli Serdang,2022-03-03
2,CUST-0002,Citra Santoso,Male,1997-11-23,citra.santoso2@example.com,8.763547e+10,Regular,SMB,Jawa Barat,Bandung,2023-07-24



products.csv


,product_id,product_name,brand,category,subcategory,unit_price,cost_price,launch_date,is_active
0,PROD-0001,Apple iPhone 16 Pro Max 256GB Desert Titanium,Apple,Smartphone,Flagship,20499000,14759000,2023-01-04,True
1,PROD-0002,Apple iPhone 16 Pro Max 512GB Natural Titanium,Apple,Smartphone,Flagship,23499000,16919000,2024-02-07,True
2,PROD-0003,Apple iPhone 16 Pro 256GB Black Titanium,Apple,Smartphone,Flagship,18499000,13319000,2025-03-10,True



stores.csv


...

,detail_id,transaction_id,product_id,promotion_id,quantity,unit_price,discount_amount
0,DTL-000001,TRX-00001,PROD-0066,PROMO-016,1,5499000,274950.0
1,DTL-000002,TRX-00001,PROD-0036,PROMO-012,1,3999000,399900.0
2,DTL-000003,TRX-00001,PROD-0098,PROMO-NONE,1,1695000,0.0



payments.csv


,payment_id,payment_method,payment_status,payment_provider,paid_amount
0,PAY-00001,QRIS,Paid,Mastercard,10518150.0
1,PAY-00002,Credit Card,Paid,OVO,6997080.0
2,PAY-00003,Debit Card,Paid,OVO,41497000.0



inventory.csv


,inventory_id,snapshot_date,store_id,product_id,stock_quantity,reorder_level,stock_status
0,INV-STORE-001-PROD-0001,2025-05-31,STORE-001,PROD-0001,38,22,Available
1,INV-STORE-001-PROD-0002,2025-05-31,STORE-001,PROD-0002,88,6,Available
2,INV-STORE-001-PROD-0003,2025-05-31,STORE-001,PROD-0003,75,18,Available
